In [245]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('data.csv ')

In [249]:
print(df.columns.tolist())

['problem_log_id', 'skill', 'problem_id', 'user_id', 'assignment_id', 'assistment_id', 'start_time', 'end_time', 'problem_type', 'original', 'correct', 'bottom_hint', 'hint_count', 'actions', 'attempt_count', 'ms_first_response', 'tutor_mode', 'sequence_id', 'student_class_id', 'position', 'type', 'base_sequence_id', 'skill_id', 'teacher_id', 'school_id', 'overlap_time', 'template_id', 'answer_id', 'answer_text', 'first_action', 'problemlogid', 'Average_confidence(FRUSTRATED)', 'Average_confidence(CONFUSED)', 'Average_confidence(CONCENTRATING)', 'Average_confidence(BORED)']


In [251]:
print(df.head)

<bound method NDFrame.head of          problem_log_id                                 skill  problem_id  \
0             137792159                                   NaN      557460   
1             138083797                              Rounding      365981   
2             142332619  Multiplication and Division Integers      426415   
3             145939397                            Proportion       86686   
4             137111284                                   NaN      399669   
...                 ...                                   ...         ...   
6123265       146114936                                   NaN      330756   
6123266       146131268                                   NaN      765964   
6123267       146118011     Addition and Subtraction Integers       84891   
6123268       146118739                                   NaN      543973   
6123269       146122458                                   NaN      136311   

         user_id  assignment_id  assistment_i

In [254]:
# 1. Convert the time columns, telling Pandas to handle mixed formats/microseconds
df['start_time'] = pd.to_datetime(df['start_time'], format='mixed')
df['end_time'] = pd.to_datetime(df['end_time'], format='mixed')

In [256]:
df['session_duration'] = (df['end_time'] - df['start_time']).dt.total_seconds()

In [257]:
df['is_scaffold'] = (df['original'] == 0).astype(int)

In [258]:
# 1. Take a 10% sample of your original data to prevent RAM crashes
# (Assuming 'df' is your main dataframe from earlier)
print(f"Original dataset size: {len(df)} rows")

Original dataset size: 6123270 rows


In [259]:
df_sampled = df.sample(frac=0.1, random_state=42).reset_index(drop=True)
print(f"Working with a sample of {len(df_sampled)} rows to build the model.")

Working with a sample of 612327 rows to build the model.


In [260]:
missing_data = df_sampled.isnull().sum()

In [261]:
# 2. Define the exact feature list (CRITICAL: 'actions' is completely removed)
final_columns = [
    'skill', 
    'correct', 
    'attempt_count',
    'hint_count',
    'bottom_hint',
    'ms_first_response',
    'first_action',
    'session_duration',
    'is_scaffold',
    'Average_confidence(FRUSTRATED)',
    'Average_confidence(CONFUSED)',
    'Average_confidence(CONCENTRATING)',
    'Average_confidence(BORED)',
    'problem_type'
]

In [262]:
# Create the final dataframe 
df_final = df_sampled[final_columns].copy()

In [263]:
# 3. Prevent Data Leakage
Y = df_final['correct'].values
X_df = df_final.drop(columns=['correct'])

In [264]:
# 4. Handle Categorical Data (One-Hot Encoding)
categorical_cols = ['skill', 'problem_type', 'first_action']
X_encoded = pd.get_dummies(X_df, columns=categorical_cols, drop_first=True)

In [265]:
#4. Strategy for other Categorical Data
categorical_cols = ['problem_type', 'first_action']
X_df[categorical_cols] = X_df[categorical_cols].fillna('Unknown')

In [266]:
print(f"Final Missing Values count: {X_df.isnull().sum().sum()}")

Final Missing Values count: 355234


In [267]:
# 5. Feature Scaling (No text columns in here!)
numerical_cols = [
    'attempt_count', 
    'hint_count', 
    'bottom_hint', 
    'ms_first_response', 
    'session_duration',
    'Average_confidence(FRUSTRATED)',
    'Average_confidence(CONFUSED)',
    'Average_confidence(CONCENTRATING)',
    'Average_confidence(BORED)'
]

In [268]:
scaler = StandardScaler()
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])

In [269]:
# 6. Memory Optimization: Downcast to 32-bit floats BEFORE converting to NumPy
# This forces everything to be a mathematical number and saves massive amounts of RAM
X_encoded = X_encoded.astype('float32')

# 7. Convert to final NumPy array
X = X_encoded.values


print(f"Shape of Input Features (X): {X.shape}")
print(f"Shape of Target Variable (Y): {Y.shape}")

Shape of Input Features (X): (612327, 211)
Shape of Target Variable (Y): (612327,)


In [270]:
print(df_final.head)

<bound method NDFrame.head of                                                 skill  correct  attempt_count  \
0                                                 NaN      1.0              1   
1                                                 NaN      1.0              1   
2                     Interpreting Coordinate Graphs       1.0              1   
3                                                 NaN      1.0              1   
4                                                 NaN      1.0              1   
...                                               ...      ...            ...   
612322  Multiplication and Division Positive Decimals      0.0              2   
612323                                            NaN      1.0              1   
612324                                            NaN      1.0              1   
612325                                            NaN      0.0              1   
612326                                           Mean      1.0              1  

In [271]:
print(df_final.columns.tolist())

['skill', 'correct', 'attempt_count', 'hint_count', 'bottom_hint', 'ms_first_response', 'first_action', 'session_duration', 'is_scaffold', 'Average_confidence(FRUSTRATED)', 'Average_confidence(CONFUSED)', 'Average_confidence(CONCENTRATING)', 'Average_confidence(BORED)', 'problem_type']


## MISSING DATA REPORT

In [273]:
import pandas as pd

# 1. Check for missing values in your current dataset (X_df)
print("--- MISSING DATA REPORT ---")

# .isnull().sum() counts the total missing values in each column
missing_data = X_df.isnull().sum()

# Filter to show ONLY columns that actually have missing data
missing_data = missing_data[missing_data > 0]

if missing_data.empty:
    print("Great news! There are 0 missing values in your dataset.")
else:
    print(missing_data)
    
    # Calculate the percentage of missing data for context
    print("\n--- PERCENTAGE OF MISSING DATA ---")
    total_rows = len(X_df)
    for col, count in missing_data.items():
        percentage = (count / total_rows) * 100
        print(f"{col}: {percentage:.2f}% missing")

--- MISSING DATA REPORT ---
skill          349212
bottom_hint      6022
dtype: int64

--- PERCENTAGE OF MISSING DATA ---
skill: 57.03% missing
bottom_hint: 0.98% missing


In [274]:
print(f"Rows before dropping missing skills: {len(X_df)}")

# 1. Drop rows where 'skill' is missing (Crucial for a Learning Path AI)
# We have to drop the matching rows in Y as well so X and Y stay perfectly aligned!
missing_skill_indices = X_df[X_df['skill'].isnull()].index

X_df = X_df.drop(index=missing_skill_indices)
# Y is a numpy array, so we use boolean indexing to keep only the valid rows
Y = Y[~np.isin(np.arange(len(Y)), missing_skill_indices)]

print(f"Rows after dropping missing skills: {len(X_df)}")

# 2. Strategy for Time/Duration: Fill with MEDIAN
time_cols = ['ms_first_response', 'session_duration']
for col in time_cols:
    median_val = X_df[col].median()
    X_df[col] = X_df[col].fillna(median_val)

# 3. Strategy for Counts and Confidence: Fill with ZERO
zero_fill_cols = [
    'attempt_count', 
    'hint_count', 
    'bottom_hint',
    'Average_confidence(FRUSTRATED)',
    'Average_confidence(CONFUSED)',
    'Average_confidence(CONCENTRATING)',
    'Average_confidence(BORED)'
]
X_df[zero_fill_cols] = X_df[zero_fill_cols].fillna(0)

# 4. Strategy for other Categorical Data
categorical_cols = ['problem_type', 'first_action']
X_df[categorical_cols] = X_df[categorical_cols].fillna('Unknown')

print("Missing values completely handled!")
print(f"Final Missing Values count: {X_df.isnull().sum().sum()}")

Rows before dropping missing skills: 612327
Rows after dropping missing skills: 263115
Missing values completely handled!
Final Missing Values count: 0


In [275]:
import numpy as np

print("--- OUTLIER HANDLING ---")

# 1. Identify the columns that actually get extreme outliers
# Confidence scores don't need this because they are usually naturally capped between 0 and 1.
outlier_cols = [
    'ms_first_response', 
    'session_duration', 
    'attempt_count', 
    'hint_count'
]

# 2. Apply Percentile Capping (Winsorization)
for col in outlier_cols:
    # Calculate the 99th percentile (the threshold for "extreme")
    upper_limit = X_df[col].quantile(0.99)
    
    # Optional: Calculate the 1st percentile for extreme low values 
    # (Useful if someone has a negative time error)
    lower_limit = X_df[col].quantile(0.01)
    
    # Cap the extreme high values
    X_df[col] = np.where(X_df[col] > upper_limit, upper_limit, X_df[col])
    
    # Cap the extreme low values
    X_df[col] = np.where(X_df[col] < lower_limit, lower_limit, X_df[col])

print("Outliers successfully capped at the 1st and 99th percentiles!")

--- OUTLIER HANDLING ---
Outliers successfully capped at the 1st and 99th percentiles!


In [276]:
print(df_final.columns.tolist())

['skill', 'correct', 'attempt_count', 'hint_count', 'bottom_hint', 'ms_first_response', 'first_action', 'session_duration', 'is_scaffold', 'Average_confidence(FRUSTRATED)', 'Average_confidence(CONFUSED)', 'Average_confidence(CONCENTRATING)', 'Average_confidence(BORED)', 'problem_type']


In [277]:
print(X_df.head)

<bound method NDFrame.head of                                                 skill  attempt_count  \
2                     Interpreting Coordinate Graphs             1.0   
9                   Addition and Subtraction Integers            1.0   
11                            Order of Operations All            1.0   
13                             Solving for a variable            1.0   
16                              Distributive Property            1.0   
...                                               ...            ...   
612316                               Finding Percents            1.0   
612318                         Greatest Common Factor            1.0   
612320                          Surface Area of Prism            1.0   
612322  Multiplication and Division Positive Decimals            2.0   
612326                                           Mean            1.0   

        hint_count  bottom_hint  ms_first_response  first_action  \
2              0.0          0.0      

In [285]:
import numpy as np

unique, counts = np.unique(Y, return_counts=True)
for val, cnt in zip(unique, counts):
    print(f"Class {val}: {cnt} samples")

Class 0.0: 79533 samples
Class 0.25: 2 samples
Class 0.5: 3 samples
Class 0.6: 1 samples
Class 0.75: 10 samples
Class 1.0: 183566 samples


In [287]:
Y_clean = (Y > 0).astype(np.float32)

unique, counts = np.unique(Y_clean, return_counts=True)
for val, cnt in zip(unique, counts):
    print(f"Class {val}: {cnt} samples")

Class 0.0: 79533 samples
Class 1.0: 183582 samples


In [289]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X_df, Y_clean, test_size=0.10, random_state=42, stratify=Y_clean)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.111, random_state=42, stratify=y_temp)

print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)

Train: (210517, 13)
Val:   (26286, 13)
Test:  (26312, 13)


In [295]:
import numpy as np

# Check what's still object
print("X_train dtypes:")
print(X_train.dtypes)
print("\nObject columns:", X_train.select_dtypes(include='object').columns.tolist())

X_train dtypes:
skill                                 object
attempt_count                        float64
hint_count                           float64
bottom_hint                          float64
ms_first_response                    float64
first_action                           int64
session_duration                     float64
is_scaffold                            int32
Average_confidence(FRUSTRATED)       float64
Average_confidence(CONFUSED)         float64
Average_confidence(CONCENTRATING)    float64
Average_confidence(BORED)            float64
problem_type                          object
dtype: object

Object columns: ['skill', 'problem_type']


In [299]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Step 1 — Re-encode directly on X_df
skill_le   = LabelEncoder()
problem_le = LabelEncoder()

X_df['skill']        = skill_le.fit_transform(X_df['skill'].astype(str))
X_df['problem_type'] = problem_le.fit_transform(X_df['problem_type'].astype(str))

# Step 2 — Convert to float32
X_df_clean = X_df.astype(np.float32)
Y_clean    = Y_clean.astype(np.float32)

# Step 3 — Verify no objects remain
print("Object columns:", X_df_clean.select_dtypes(include='object').columns.tolist())
print("X_df dtypes unique:", X_df_clean.dtypes.unique())
print("Y dtype:", Y_clean.dtype)
print("Shapes:", X_df_clean.shape, Y_clean.shape)

Object columns: []
X_df dtypes unique: [dtype('float32')]
Y dtype: float32
Shapes: (263115, 13) (263115,)


In [301]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X_df_clean, Y_clean, test_size=0.10, random_state=42, stratify=Y_clean)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.111, random_state=42, stratify=y_temp)

print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)

Train: (210517, 13)
Val:   (26286, 13)
Test:  (26312, 13)


In [303]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100


C:\Users\ASUS\anaconda3\envs\ocr-numpy1\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


6579/6579 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.6749 - loss: 12.7018 - val_accuracy: 0.6977 - val_loss: 0.6128
Epoch 2/100
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.6976 - loss: 0.6213 - val_accuracy: 0.6977 - val_loss: 0.6128
Epoch 3/100
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.6977 - loss: 0.6168 - val_accuracy: 0.6977 - val_loss: 0.6128
Epoch 4/100
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.6977 - loss: 0.6197 - val_accuracy: 0.6977 - val_loss: 0.6128
Epoch 5/100
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.6977 - loss: 0.6237 - val_accuracy: 0.6977 - val_loss: 0.6128
Epoch 6/100
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.6977 - loss: 0.6163 - val_accuracy: 0.6977 - val_loss: 0.6129
Epoch 7/100
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.6977 - loss: 0.6130 - val_accuracy: 0.6977 - val_loss: 0.6128
Epoch 8/100
6579/6579 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.6977 - loss: 0.6129 - val

In [305]:
#Confirm the class imbalance
import numpy as np
zeros = np.sum(Y_clean == 0)
ones  = np.sum(Y_clean == 1)
print(f"Class 0 (incorrect): {zeros} ({zeros/len(Y_clean)*100:.1f}%)")
print(f"Class 1 (correct):   {ones}  ({ones/len(Y_clean)*100:.1f}%)")

Class 0 (incorrect): 79533 (30.2%)
Class 1 (correct):   183582  (69.8%)


In [307]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Class weights to penalize missing the minority class
total = len(y_train)
n_zeros = np.sum(y_train == 0)
n_ones  = np.sum(y_train == 1)

class_weight = {
    0: total / (2 * n_zeros),
    1: total / (2 * n_ones)
}
print("Class weights:", class_weight)

# Rebuild model with BatchNormalization
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Class weights: {0: 1.6541235817330358, 1: 0.7166145843971051}


C:\Users\ASUS\anaconda3\envs\ocr-numpy1\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                     │ (None, 128)                 │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_10 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_21 (Dense)                     │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 64)                  │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_11 (Dropout)                 │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_22 (Dense)                     │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_23 (Dense)                     │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 12,929 (50.50 KB)

 Trainable params: 12,545 (49.00 KB)

 Non-trainable params: 384 (1.50 KB)

In [309]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
3290/3290 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.5888 - loss: 0.6816 - val_accuracy: 0.5885 - val_loss: 0.6837
Epoch 2/100
3290/3290 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.6078 - loss: 0.6779 - val_accuracy: 0.4954 - val_loss: 0.7639
Epoch 3/100
3290/3290 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.6834 - loss: 0.6021 - val_accuracy: 0.5854 - val_loss: 0.6585
Epoch 4/100
3290/3290 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.7517 - loss: 0.5198 - val_accuracy: 0.7868 - val_loss: 0.5412
Epoch 5/100
3290/3290 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7608 - loss: 0.5110 - val_accuracy: 0.4829 - val_loss: 0.7392
Epoch 6/100
3290/3290 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.7745 - loss: 0.5034 - val_accuracy: 0.5680 - val_loss: 0.6955
Epoch 7/100
3290/3290 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.7183 - loss: 0.5872 - val_accuracy: 0.6827 - val_loss: 0.6411
Epoch 8/100
3290/3290 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.7688 - loss: 0